# Validation Data Pipeline (No Balancing, No Tricks)

Key rule:
- Validation pipeline uses the same preprocessing as training.
- No balancing, no class weighting, no oversampling, no undersampling.

Expected output structure:
processed_dataset/
  train/
    caries/
    no_caries/
  val/
    caries/
    no_caries/

In [1]:
import os
import json
import cv2
import numpy as np

BASE_PATH = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/validation_data/quadrant_enumeration_disease"
IMG_FOLDER = BASE_PATH + "/xrays"
JSON_PATH = BASE_PATH + "/validation_triple.json"

print('IMG_FOLDER exists:', os.path.exists(IMG_FOLDER))
print('JSON_PATH exists:', os.path.exists(JSON_PATH))

IMG_FOLDER exists: True
JSON_PATH exists: True


In [2]:
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

images = data.get('images', [])
annotations = data.get('annotations', [])
categories_3 = data.get('categories_3', [])

image_id_to_file = {img['id']: img['file_name'] for img in images}

print('images:', len(images))
print('annotations:', len(annotations))

images: 50
annotations: 182


In [3]:
def ann_to_binary_label(ann):
    c3 = ann.get('category_id_3', -1)
    if c3 in [1, 3]:
        return 'caries'
    else:
        return 'no_caries'

In [4]:
OUTPUT_BASE = 'processed_dataset'

val_caries_dir = os.path.join(OUTPUT_BASE, 'val', 'caries')
val_no_caries_dir = os.path.join(OUTPUT_BASE, 'val', 'no_caries')

os.makedirs(val_caries_dir, exist_ok=True)
os.makedirs(val_no_caries_dir, exist_ok=True)

saved_counts = {'caries': 0, 'no_caries': 0}
skipped = 0

for idx, ann in enumerate(annotations):
    img_id = ann.get('image_id')
    file_name = image_id_to_file.get(img_id)
    bbox = ann.get('bbox')

    if file_name is None or bbox is None or len(bbox) != 4:
        skipped += 1
        continue

    image_path = os.path.join(IMG_FOLDER, file_name)
    img = cv2.imread(image_path)
    if img is None:
        skipped += 1
        continue

    x, y, w, h = bbox
    x1 = max(int(np.floor(x)), 0)
    y1 = max(int(np.floor(y)), 0)
    x2 = min(int(np.ceil(x + w)), img.shape[1])
    y2 = min(int(np.ceil(y + h)), img.shape[0])

    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        skipped += 1
        continue

    crop = cv2.resize(crop, (224, 224))

    label = ann_to_binary_label(ann)

    save_dir = os.path.join(OUTPUT_BASE, 'val', label)
    os.makedirs(save_dir, exist_ok=True)

    out_name = f"{img_id}_{ann.get('id', idx)}.png"
    out_path = os.path.join(save_dir, out_name)

    if cv2.imwrite(out_path, crop):
        saved_counts[label] += 1
    else:
        skipped += 1

print('Validation Saved:', saved_counts)
print('Skipped:', skipped)

Validation Saved: {'caries': 133, 'no_caries': 49}
Skipped: 0


## Critical Check (Do Not Skip)
1. Open a few generated images manually from val/caries and val/no_caries.
2. Confirm crops are correct and teeth are visible.
3. Observe class balance only. Do not fix imbalance here.

In [5]:
print('Validation Saved Summary:')
print('caries =', saved_counts['caries'])
print('no_caries =', saved_counts['no_caries'])

Validation Saved Summary:
caries = 133
no_caries = 49
